# 🔧 Notebook 00 — Environment Setup & Verification

**SLM Fine-Tuning Tutorial** | Notebook 0 of 19

---

## What This Notebook Covers

Before diving into fine-tuning techniques, we need to verify that the environment is properly configured. This notebook:

1. **Verifies all required packages** are installed with correct versions
2. **Detects hardware** — Apple Silicon (MPS), NVIDIA GPU (CUDA), or CPU
3. **Profiles available memory** for planning which models/techniques fit
4. **Downloads a test model** (SmolLM2-135M-Instruct) and runs quick inference
5. **Benchmarks CPU vs GPU** inference to confirm acceleration is working
6. **Previews the learning path** — a roadmap of all techniques we'll cover

### Supported Hardware

| Tier | Hardware | What Works |
|------|----------|------------|
| 6 GB GPU | RTX 3060, Colab T4 (free) | SmolLM2-135M (all techniques) |
| 16 GB | Apple M1/M2, RTX 4060 Ti | SmolLM2-360M (LoRA/DoRA) |
| 36 GB | Apple M3/M4, RTX 3090 | All notebooks incl. Qwen2.5-1.5B LoRA |
| 64+ GB | Apple M-series (64GB+), A100 | Everything at full precision |

> **Minimum**: 6 GB GPU (or CPU-only for concept understanding). No student is locked out of any concept.

In [ ]:
# ============================================================
# COLAB SETUP — Uncomment the lines below if running on Google Colab
# ============================================================
# !pip install -q torch transformers peft trl datasets accelerate
# !pip install -q bitsandbytes evaluate rouge-score bert-score
# !pip install -q matplotlib plotly scikit-learn safetensors
#
# # Clone the repo to get src/ utilities
# # !git clone https://github.com/<your-username>/finetuning-slm.git
# # %cd finetuning-slm/notebooks
#
# # Mount Google Drive for checkpoint persistence
# # from google.colab import drive
# # drive.mount('/content/drive')

print("Colab setup cell — uncomment the lines above if running in Google Colab.")
print("For local environments, all packages should already be installed.")

## 1. Package Verification

Let's verify that all required packages are installed and check their versions. This helps catch environment issues early.

In [ ]:
import sys
import os
from pathlib import Path

# Make src/ importable from notebooks/
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")
print()

# --- Kernel Verification ---
import IPython
kernel_name = IPython.get_ipython().kernel.session.key if hasattr(IPython.get_ipython(), 'kernel') else "unknown"
expected_kernels = ["finetuning-slm", "agentic-slm"]
try:
    kernel_info = !jupyter kernelspec list 2>/dev/null | grep -E "finetuning-slm|agentic-slm"
    if kernel_info:
        print(f"✅ Kernel detected: {kernel_info[0].strip()}")
    else:
        print("⚠️  Expected kernel 'finetuning-slm' not found. You may be running in a different environment.")
        print("   This is fine for Colab — all cells will still work.")
except:
    print("ℹ️  Kernel check skipped (non-Jupyter environment)")
print()


# --- Package Verification ---
REQUIRED_PACKAGES = [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("peft", "peft"),
    ("trl", "trl"),
    ("datasets", "datasets"),
    ("accelerate", "accelerate"),
    ("bitsandbytes", "bitsandbytes"),
    ("evaluate", "evaluate"),
    ("rouge_score", "rouge-score"),
    ("bert_score", "bert-score"),
    ("matplotlib", "matplotlib"),
    ("plotly", "plotly"),
    ("sklearn", "scikit-learn"),
]

print(f"{'Package':<20} {'Version':<15} {'Status'}")
print("─" * 50)

all_ok = True
for import_name, display_name in REQUIRED_PACKAGES:
    try:
        mod = __import__(import_name)
        version = getattr(mod, "__version__", "installed")
        print(f"{display_name:<20} {version:<15} ✅")
    except ImportError:
        print(f"{display_name:<20} {'—':<15} ❌ NOT FOUND")
        all_ok = False

print("─" * 50)
if all_ok:
    print("\n✅ All required packages are installed!")
else:
    print("\n⚠️  Some packages are missing. Attempting to install...")
    import subprocess
    missing = [disp for imp, disp in REQUIRED_PACKAGES if not __import__("importlib").util.find_spec(imp)]
    for pkg in missing:
        print(f"   Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print("   ✅ Missing packages installed. Please restart the kernel and re-run.")


## 2. Hardware Detection

We detect whether you have:
- **Apple Silicon (MPS)** — M1/M2/M3/M4 with Metal Performance Shaders
- **NVIDIA GPU (CUDA)** — dedicated GPU with CUDA support
- **CPU only** — still works, just slower

This uses the `src/training_utils.py` and `src/apple_silicon.py` utility modules.

In [ ]:
import platform
import torch

from src.training_utils import setup_device, get_device_info
from src.apple_silicon import check_mps_availability

# --- Device Detection ---
device = setup_device()
device_info = get_device_info()
system_info = check_mps_availability()

print("╔══════════════════════════════════════════════════╗")
print("║           SYSTEM & HARDWARE INFO                ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Platform:    {system_info['platform'][:34]:<34} ║")
print(f"║  Chip:        {system_info.get('chip', 'N/A')[:34]:<34} ║")

ram_str = f"{system_info.get('system_memory_gb', 'N/A'):.0f}" if isinstance(system_info.get('system_memory_gb'), (int, float)) else "N/A"
print(f"║  RAM:         {ram_str + ' GB':<34} ║")
print(f"║  Python:      {system_info['python_version']:<34} ║")
print(f"║  PyTorch:     {system_info['pytorch_version']:<34} ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Device:      {str(device):<34} ║")
print(f"║  Backend:     {device_info['backend'][:34]:<34} ║")
print(f"║  MPS Built:   {str(system_info['mps_built']):<34} ║")
print(f"║  MPS Avail:   {str(system_info['mps_available']):<34} ║")
if device.type == "cuda":
    print(f"║  VRAM:        {device_info.get('vram_gb', 0):.1f} GB{'':<28} ║")
print("╚══════════════════════════════════════════════════╝")

# Expected output (Apple M4 Max 36GB):
# Device: mps, Backend: Metal Performance Shaders (Apple Silicon)
# Expected output (Colab T4):
# Device: cuda, Backend: CUDA (Tesla T4), VRAM: 15.0 GB

## 3. Memory Profiling

Understanding available memory helps us choose the right model size and technique. Key thresholds:

| Available Memory | What Fits |
|-----|-----|
| 4–6 GB | SmolLM2-135M (any technique) |
| 8–16 GB | SmolLM2-360M LoRA, Qwen2.5-1.5B QLoRA |
| 16–36 GB | Qwen2.5-1.5B LoRA, full finetune of 135M |
| 36+ GB | Full finetune up to 1.5B, distillation with 3B teacher |

In [ ]:
from src.apple_silicon import get_memory_stats

# --- Memory Profiling ---
memory_stats = get_memory_stats()

print("\n📊 Memory Status")
print("─" * 40)

if memory_stats["backend"] == "MPS":
    # MPS uses unified memory — report system RAM as available pool
    print(f"  Backend:             MPS (Unified Memory)")
    print(f"  Currently Allocated: {memory_stats['allocated_gb']:.3f} GB")
    print(f"  Driver Allocated:    {memory_stats['driver_allocated_gb']:.3f} GB")
    # System RAM is the effective "VRAM" on Apple Silicon
    try:
        import subprocess
        result = subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True)
        if result.returncode == 0:
            total_ram_gb = int(result.stdout.strip()) / (1024**3)
            # ~75% of unified memory is typically available for ML
            effective_gb = total_ram_gb * 0.75
            print(f"  Total System RAM:    {total_ram_gb:.0f} GB")
            print(f"  Effective for ML:    ~{effective_gb:.0f} GB (75% of unified memory)")
    except Exception:
        pass

elif memory_stats["backend"] == "CUDA":
    print(f"  Backend:             CUDA")
    print(f"  Allocated:           {memory_stats['allocated_gb']:.3f} GB")
    print(f"  Reserved:            {memory_stats['reserved_gb']:.3f} GB")
    total_vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    free_vram = total_vram - memory_stats['allocated_gb']
    print(f"  Total VRAM:          {total_vram:.1f} GB")
    print(f"  Free VRAM:           {free_vram:.1f} GB")

else:
    print(f"  Backend:             CPU (no GPU acceleration)")
    try:
        import psutil
        ram = psutil.virtual_memory()
        print(f"  Total RAM:           {ram.total / 1e9:.1f} GB")
        print(f"  Available RAM:       {ram.available / 1e9:.1f} GB")
    except ImportError:
        print("  (Install 'psutil' for detailed RAM info)")

print("─" * 40)

# Quick tensor test on device
print("\n🧪 Quick Tensor Test")
x = torch.randn(1000, 1000, device=device)
y = torch.randn(1000, 1000, device=device)
z = x @ y  # Matrix multiply
print(f"  1000×1000 matmul on {device}: shape={z.shape}, dtype={z.dtype}")
print("  ✅ Tensor operations working on device!")
del x, y, z

# Expected output (MPS):
# Backend: MPS (Unified Memory), Total System RAM: 36 GB, Effective for ML: ~27 GB
# Expected output (CUDA T4):
# Backend: CUDA, Total VRAM: 15.0 GB, Free VRAM: ~14.5 GB

# --- Additional Memory Guidance ---
import psutil
total_ram_gb = psutil.virtual_memory().total / (1024**3) if 'psutil' in dir() else None

if total_ram_gb is None:
    try:
        import subprocess
        result = subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True)
        total_ram_gb = int(result.stdout.strip()) / (1024**3) if result.returncode == 0 else None
    except:
        total_ram_gb = None

if total_ram_gb:
    ml_usable = total_ram_gb * 0.75  # ~75% is safely usable for ML
    print(f"\n📐 Memory Budget Guidance:")
    print(f"   Total System RAM:     {total_ram_gb:.0f} GB")
    print(f"   Estimated ML Budget:  {ml_usable:.0f} GB (75% of total)")
    print(f"   ┣━ SmolLM2-135M fp16: ~0.3 GB  ✅")
    print(f"   ┣━ SmolLM2-360M fp16: ~0.7 GB  {'✅' if ml_usable > 2 else '⚠️'}")
    print(f"   ┣━ Qwen2.5-1.5B QLoRA: ~2.8 GB {'✅' if ml_usable > 6 else '⚠️'}")
    print(f"   ┗━ Qwen2.5-3B fp16:   ~6.0 GB  {'✅' if ml_usable > 12 else '❌'}")



## 4. Model Download & Quick Inference Test

We'll download **SmolLM2-135M-Instruct** — our smallest model — and verify that inference works correctly. This model is only ~270 MB and fits on any hardware.

> **First run**: The model will be downloaded from HuggingFace Hub (~270 MB). Subsequent runs use the cache.

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

print(f"Loading model: {MODEL_ID}")
print("(First run downloads ~270 MB from HuggingFace Hub)\n")

# --- Load Model & Tokenizer ---
t0 = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,  # Use fp32 for maximum compatibility
)

load_time = time.perf_counter() - t0
print(f"✅ Model loaded in {load_time:.1f}s")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   Size: ~{model.num_parameters() * 4 / 1e9:.2f} GB (fp32)")

# Move to device
model = model.to(device)
print(f"   Device: {next(model.parameters()).device}")

# --- Quick Inference Test ---
print("\n" + "─" * 50)
print("📝 Quick Inference Test")
print("─" * 50)

messages = [
    {"role": "user", "content": "What is fine-tuning in machine learning? Explain in 2 sentences."}
]

# Apply chat template
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(device)

# Generate
t0 = time.perf_counter()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
gen_time = time.perf_counter() - t0

# Decode only the new tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

num_tokens = len(generated_tokens)
tokens_per_sec = num_tokens / gen_time

print(f"\nPrompt: \"What is fine-tuning in machine learning?\"")
print(f"\nResponse: {response}")
print(f"\n⏱️  Generated {num_tokens} tokens in {gen_time:.2f}s ({tokens_per_sec:.1f} tok/s)")
print("\n✅ Inference is working!")

## 5. CPU vs GPU Inference Benchmark

Let's measure the actual speedup from GPU acceleration. This confirms that MPS/CUDA is working properly and gives you a baseline for training performance expectations.

> **Note**: On CPU-only systems, both runs will be on CPU (benchmark still runs for consistency).

In [ ]:
import matplotlib.pyplot as plt

BENCHMARK_PROMPT = "Explain the difference between supervised and unsupervised learning."
NUM_TOKENS = 50


def benchmark_inference(model, tokenizer, device_target, prompt, max_new_tokens):
    """Run inference on a specific device and measure time."""
    model_on_device = model.to(device_target)
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(device_target)

    # Warmup
    with torch.no_grad():
        _ = model_on_device.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)

    # Timed run
    if device_target.type == "mps":
        torch.mps.synchronize()
    elif device_target.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model_on_device.generate(
            **inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id,
            do_sample=False,  # Greedy for reproducibility
        )

    if device_target.type == "mps":
        torch.mps.synchronize()
    elif device_target.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0
    actual_new_tokens = outputs.shape[1] - inputs["input_ids"].shape[1]
    return elapsed, actual_new_tokens


# --- Run Benchmarks ---
print("⏱️  Running CPU vs GPU benchmark...\n")

# CPU benchmark
cpu_device = torch.device("cpu")
cpu_time, cpu_tokens = benchmark_inference(model, tokenizer, cpu_device, BENCHMARK_PROMPT, NUM_TOKENS)
cpu_tps = cpu_tokens / cpu_time
print(f"  CPU:  {cpu_time:.2f}s for {cpu_tokens} tokens ({cpu_tps:.1f} tok/s)")

# GPU benchmark (if available)
gpu_label = device.type.upper()
if device.type in ("mps", "cuda"):
    gpu_time, gpu_tokens = benchmark_inference(model, tokenizer, device, BENCHMARK_PROMPT, NUM_TOKENS)
    gpu_tps = gpu_tokens / gpu_time
    print(f"  {gpu_label}:  {gpu_time:.2f}s for {gpu_tokens} tokens ({gpu_tps:.1f} tok/s)")
    speedup = cpu_time / gpu_time
    print(f"\n  🚀 Speedup: {speedup:.1f}x faster on {gpu_label}")
else:
    gpu_time = cpu_time
    gpu_tps = cpu_tps
    gpu_label = "CPU (no GPU)"
    speedup = 1.0
    print("  ⚠️  No GPU available — both benchmarks ran on CPU")

# --- Visualization ---

# --- Benchmark Results Table ---
print("\n" + "═" * 60)
print(f"{'Device':<12} {'Time (s)':<12} {'Tokens':<10} {'Tok/s':<10} {'Speedup'}")
print("─" * 60)
print(f"{'CPU':<12} {cpu_time:<12.3f} {num_tokens:<10} {cpu_tps:<10.1f} {'1.0x (baseline)'}")
if gpu_time > 0:
    print(f"{device.type.upper():<12} {gpu_time:<12.3f} {num_tokens:<10} {gpu_tps:<10.1f} {f'{speedup:.1f}x'}")
else:
    print(f"{'(no GPU)':<12} {'—':<12} {'—':<10} {'—':<10} {'N/A'}")
print("═" * 60)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Time comparison
bars1 = axes[0].bar(["CPU", gpu_label], [cpu_time, gpu_time],
                    color=["#4a90d9", "#e74c3c"], edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Time (seconds)")
axes[0].set_title(f"Inference Time ({NUM_TOKENS} tokens)")
axes[0].bar_label(bars1, fmt="%.2fs")

# Tokens per second
bars2 = axes[1].bar(["CPU", gpu_label], [cpu_tps, gpu_tps],
                    color=["#4a90d9", "#e74c3c"], edgecolor="black", linewidth=0.5)
axes[1].set_ylabel("Tokens / second")
axes[1].set_title(f"Throughput (speedup: {speedup:.1f}x)")
axes[1].bar_label(bars2, fmt="%.1f")

plt.tight_layout()

# Save chart
os.makedirs("../outputs", exist_ok=True)
plt.savefig("../outputs/00_benchmark.png", dpi=100, bbox_inches="tight")
plt.show()
print("\n📊 Benchmark chart saved to outputs/00_benchmark.png")

# Expected output (M4 Max):
# CPU: ~3-5s, MPS: ~1-2s, Speedup: ~2-3x
# Expected output (Colab T4):
# CPU: ~5-8s, CUDA: ~0.5-1s, Speedup: ~5-10x



## 6. What's Ahead — Choosing a Fine-Tuning Method

This tutorial covers **10+ fine-tuning techniques** from basic to advanced. Here's a roadmap of what each notebook covers and when to use each method.

### Quick Decision Guide

- **Have lots of data + compute?** → Full Fine-Tuning (Notebook 02)
- **Want cheap task adaptation?** → LoRA / DoRA (Notebook 04)
- **Only 6 GB VRAM?** → QLoRA (Notebook 05)
- **Don't want to touch weights at all?** → Prompt Tuning (Notebook 06)
- **Need to align with human preferences?** → DPO (Notebook 08)
- **Want to compress a large model?** → Knowledge Distillation (Notebook 09)

In [ ]:
# --- Technique Overview Table ---

techniques = [
    ("02", "Full Fine-Tuning", "100%", "High", "Large data + compute", "Maximum adaptation"),
    ("03", "Feature Extraction", "<1%", "Very Low", "Quick prototyping", "Freeze all, train head"),
    ("04", "LoRA + DoRA", "0.1–4%", "Low", "Most common starting point", "Low-rank weight updates"),
    ("05", "QLoRA", "0.1–4%", "Very Low", "6 GB GPU / Colab free tier", "4-bit base + LoRA"),
    ("06", "Prompt Tuning", "<0.01%", "Minimal", "Multi-task, no weight edits", "Learned soft prompts"),
    ("07", "Prefix Tuning", "<0.1%", "Minimal", "Generation tasks", "Learned prefix activations"),
    ("08", "DPO Alignment", "Varies", "Medium", "Safety, style, preferences", "Direct preference optimization"),
    ("09", "Knowledge Distillation", "100% (student)", "High", "Deploy smaller model", "Teacher→Student transfer"),
    ("10", "Model Merging", "0%", "None", "Combine specialists", "Weight averaging/TIES"),
]

print("\n" + "═" * 95)
print(f"{'NB':<4} {'Technique':<25} {'Params Trained':<15} {'Memory':<10} {'Best For':<28} {'How It Works'}")
print("═" * 95)
for nb, tech, params, mem, best_for, how in techniques:
    print(f"{nb:<4} {tech:<25} {params:<15} {mem:<10} {best_for:<28} {how}")
print("═" * 95)

print("\n📚 Additional Notebooks:")
print("   11 — Failure Points & Debugging")
print("   12 — Responsible & Safe Fine-Tuning")
print("   13 — Evaluation Theory & Benchmarking")
print("   14–16 — Fun Use Cases (Shakespeare, Code Roast, Dungeon Master)")
print("   17–18 — Industry Use Cases (Agentic Tool Calling, Clinical Summarizer)")
print("   19 — Apple Silicon Optimization Guide")

## Next Steps

Your environment is verified and ready! Continue to:

👉 **[Notebook 01 — Dataset Design & Data Preparation](01_dataset_design_and_data_prep.ipynb)**

In Notebook 01 you'll learn:
- How to choose and format datasets for different fine-tuning objectives
- Chat template formatting (ChatML, Alpaca, custom)
- Data quality checks and preprocessing pipelines
- How to create your own instruction-tuning dataset

---

### Cleanup

In [ ]:
# Free memory from the model we loaded
del model
del tokenizer

if device.type == "mps":
    torch.mps.empty_cache()
elif device.type == "cuda":
    torch.cuda.empty_cache()

import gc
gc.collect()

print("🧹 Memory cleared. Ready for the next notebook!")